# Figure 5b - MATRIX

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\StorywranglerData\RussiaUkraineJsons_19092023'
os.chdir(directory)

remove_UA_RU=False
smooth=True
Filter=False

In [ ]:
df = pd.read_csv(r'pivotUkraine_SUM_excelInput_weekly.csv') #, index_col=0, sep=';'
#Load frequency share pivoted df
df.set_index(df.columns[0], inplace=True)
df.head(3)

In [ ]:
# Filter out non-date columns: keep only columns that can be parsed as dates.
date_mask = pd.to_datetime(df.columns, errors='coerce').notna()
date_cols = df.columns[date_mask]
df_dates = df[date_cols].copy()

# Convert the remaining column names to datetime objects
df_dates.columns = pd.to_datetime(df_dates.columns)

# At this point, df_dates has languages as rows and dates as columns.
# To compute 6‑week sums, transpose so that dates become the index.
df_transposed = df_dates.T

# Group the rows (dates) by 6‑week periods and sum the values.
# Here, label='right' and closed='right' make the group label the last date of each 6‑week period.
df_6week_transposed = df_transposed.groupby(
    pd.Grouper(freq='6W', label='right', closed='right')
).sum()

# Transpose back so that rows are languages and columns represent 6‑week aggregated sums.
df = df_6week_transposed.T


# Matrix

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from scipy.spatial.distance import cdist  # for jensenshannon

# Assume 'df' is your DataFrame with languages as rows and dates as columns,
# with one extra column (e.g., "language") that should be excluded.
# Filter out non-date columns:
date_mask = pd.to_datetime(df.columns, errors='coerce').notna()
date_cols = df.columns[date_mask]
df_dates = df[date_cols].copy()

# Convert the remaining column names to datetime objects
df_dates.columns = pd.to_datetime(df_dates.columns)

# Transpose the DataFrame so that each date becomes an observation (vector)
# Now, the index are dates and columns are languages.
df_transposed = df_dates.T



# Plotly

In [ ]:
# Optionally, if using jensenshannon, ensure each row sums to 1:
# df_transposed = df_transposed.div(df_transposed.sum(axis=1), axis=0)

# Choose the distance metric: set metric to 'cosine', 'euclidean', or 'jensenshannon'
metric = 'jensenshannon'  # Change as needed

# Compute distance matrix based on the chosen metric
if metric == 'cosine':
    distance_matrix = cosine_distances(df_transposed.values)
elif metric == 'euclidean':
    distance_matrix = euclidean_distances(df_transposed.values)
elif metric == 'jensenshannon':
    distance_matrix = cdist(df_transposed.values, df_transposed.values, metric='jensenshannon')
else:
    raise ValueError("Metric must be 'cosine', 'euclidean', or 'jensenshannon'.")

# Create a DataFrame for the distance matrix; use dates as labels
distance_df = pd.DataFrame(distance_matrix, index=df_transposed.index, columns=df_transposed.index)


In [ ]:
import numpy as np

# Transform the data using log10.
# Make sure your distance_df.values are all positive; if needed, add a small constant.
z_log = np.log10(distance_df.values)

# Create the heatmap with the log-transformed data.
fig = go.Figure(data=go.Heatmap(
    z=z_log,
    x=distance_df.columns,  # datetime objects for the x-axis
    y=distance_df.index,    # datetime objects for the y-axis
    colorscale='gray',
    colorbar=dict(
        title=f'{metric.capitalize()}<br>Log Distance',
        # Define tick values for the log scale and convert them back to original scale for display.
        tickvals=np.linspace(z_log.min(), z_log.max(), 5),
        ticktext=[f"{10**v:.2f}" for v in np.linspace(z_log.min(), z_log.max(), 5)]
    )
))

# Generate tick marks for each year present in the data.
dates = pd.to_datetime(distance_df.index)
unique_years = sorted(set(dates.year))
tickvals = []
ticktext = []
for year in unique_years:
    year_dates = dates[dates.year == year]
    if not year_dates.empty:
        tick_val = min(year_dates)
        tickvals.append(tick_val)
        ticktext.append(str(year))

# Update x-axis and y-axis to show tick marks for each year.
fig.update_xaxes(
    tickmode='array',
    tickvals=tickvals,
    ticktext=ticktext,
    type='date'
)
fig.update_yaxes(
    tickmode='array',
    tickvals=tickvals,
    ticktext=ticktext,
    type='date'
)

# Update layout with custom dimensions 700x700 pixels.
fig.update_layout(
    title=f'{metric.capitalize()} Distance Heatmap of Date Vectors',
    xaxis_title='Date',
    yaxis_title='Date',
    template='plotly_white',
    width=700,
    height=700
)

fig.show()


# Matplotlib saving

In [ ]:
# Optionally, if using jensenshannon, ensure each row sums to 1:
# df_transposed = df_transposed.div(df_transposed.sum(axis=1), axis=0)

# Choose the distance metric: set metric to 'cosine', 'euclidean', or 'jensenshannon'
metric = 'JSD'  # Change as needed

# Compute distance matrix based on the chosen metric
if metric == 'cosine':
    distance_matrix = cosine_distances(df_transposed.values)
elif metric == 'euclidean':
    distance_matrix = euclidean_distances(df_transposed.values)
elif metric == 'JSD':
    distance_matrix = cdist(df_transposed.values, df_transposed.values, metric='jensenshannon')
else:
    raise ValueError("Metric must be 'cosine', 'euclidean', or 'jensenshannon'.")

# Create a DataFrame for the distance matrix; use dates as labels
distance_df = pd.DataFrame(distance_matrix, index=df_transposed.index, columns=df_transposed.index)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Assume distance_df is your DataFrame with datetime index (rows) and columns,
# and that 'metric' is defined (e.g., metric = 'cosine').
# Ensure no zero or negative values before taking log; add a small constant if needed.
constant = 1e-10

# Create a boolean mask for the diagonal
mask_diag = np.eye(distance_df.shape[0], dtype=bool)

# Extract the values and only apply epsilon to diagonal entries.
z = distance_df.values
# For diagonal elements, add epsilon to avoid log(0); leave all others unchanged.
z_log = np.log10(np.where(mask_diag, z + constant, z))

# Mask the diagonal so that its (small or dummy) values do not affect the colormap scaling.
z_log_masked = np.ma.masked_array(z_log, mask=mask_diag)

# Compute vmin and vmax ignoring the masked diagonal.
vmin = z_log_masked.min()
vmax = z_log_masked.max()

# Determine the numeric date bounds for the heatmap axes.
x_min = mdates.date2num(distance_df.columns.min())
x_max = mdates.date2num(distance_df.columns.max())
y_min = mdates.date2num(distance_df.index.min())
y_max = mdates.date2num(distance_df.index.max())

# Create the figure with desired dimensions.
fig, ax = plt.subplots(figsize=(7, 7))

# Use a copy of the gray colormap and set its "bad" color to black.
cmap_gray = plt.get_cmap('gray').copy()#.reversed()
cmap_gray.set_bad('black')

# Display the heatmap using the masked array.
im = ax.imshow(z_log_masked, aspect='auto', origin='lower', cmap=cmap_gray,
               extent=[x_min, x_max, y_min, y_max], vmin=vmin, vmax=vmax)

# Primary colorbar for the heatmap (using the log-scaled gray colormap).
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
tick_vals = np.linspace(vmin, vmax, 5)
tick_labels = [f"{10**v:.2f}" for v in tick_vals]
tick_labels[0] = "0"  # manually set the minimum tick to 0

cbar.set_ticks(tick_vals)
cbar.set_ticklabels(tick_labels)
cbar.ax.tick_params(labelsize=10)
cbar.ax.set_title(f'{metric} \nDistance', fontsize=10)

# Convert axes to date axes.
ax.xaxis_date()
ax.yaxis_date()

# Get unique years from the data (removing 2008).
unique_years = sorted(set(distance_df.columns.year))
unique_years = [year for year in unique_years if year != 2008]
tick_dates = [pd.Timestamp(year=year, month=1, day=1) for year in unique_years]
tick_positions = [mdates.date2num(tick) for tick in tick_dates]
tick_labels = [tick.strftime("%Y") for tick in tick_dates]

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, fontsize=11, rotation=25)
ax.set_yticks(tick_positions)
ax.set_yticklabels(tick_labels, fontsize=11)

# Set title.
ax.set_title(f'{metric.capitalize()} Distance Heatmap of Date Vectors', fontsize=14)
# Keep axes equal.
ax.set_aspect('equal', adjustable='box')

# ----- Add "axis-following" viridis colorbars along the top and right borders -----
divider = make_axes_locatable(ax)

# Top colorbar for the x-axis.
cax_top = divider.append_axes("top", size="5%", pad=0.05)
sm_x = plt.cm.ScalarMappable(cmap='viridis', norm=mcolors.Normalize(vmin=x_min, vmax=x_max))
sm_x.set_array([])
cbar_top = plt.colorbar(sm_x, cax=cax_top, orientation='horizontal')
# Remove ticks and labels.
cbar_top.set_ticks([])
cbar_top.ax.tick_params(labelsize=0)
cax_top.xaxis.set_ticks_position("top")
cax_top.xaxis.set_label_position("top")

# Right colorbar for the y-axis.
cax_right = divider.append_axes("right", size="5%", pad=0.05)
sm_y = plt.cm.ScalarMappable(cmap='viridis', norm=mcolors.Normalize(vmin=y_min, vmax=y_max))
sm_y.set_array([])
cbar_right = plt.colorbar(sm_y, cax=cax_right, orientation='vertical')
# Remove ticks and labels.
cbar_right.set_ticks([])
cbar_right.ax.tick_params(labelsize=0)

plt.tight_layout()
plt.show()


In [ ]:
import os
from datetime import datetime

# Define the base output directory (modify this path as needed)
base_output_dir = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter\Visualizations\Fig.5_DateVectors\matrix\\'

# Get current date and time formatted as YearMonthDay_HHMMSS (e.g., 20250314_152530)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create a new subdirectory under the base output directory using the timestamp
output_dir = os.path.join(base_output_dir, timestamp + os.sep)

# Create subdirectories for saving the files inside the timestamped directory
os.makedirs(os.path.join(output_dir, "pdf"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "jpeg"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "svg"), exist_ok=True)

# Use the timestamp in your filename; ensure 'metric' is defined or replace it with a string
#today_str = datetime.now().strftime("%Y%m%d")
fig.savefig(os.path.join(output_dir, "pdf", f"totalfreq_{metric}_{timestamp}.pdf"), format='pdf', bbox_inches='tight')
fig.savefig(os.path.join(output_dir, "jpeg", f"totalfreq_{metric}_{timestamp}.jpg"), format='jpg', dpi=500, bbox_inches='tight')
fig.savefig(os.path.join(output_dir, "svg", f"totalfreq_{metric}_{timestamp}.svg"), format='svg', bbox_inches='tight')
